# Recurrent Energy NodeField ablation

Thin driver for the repository implementation. Default execution runs only the seed-0 smoke matrix (100 graphs, ten epochs). Set `RUN_FULL = True` to train five seeds on 1,000 graphs and run the complete grid. Every execution creates a new result directory. The driver uses CUDA when available, then Apple MPS, and otherwise CPU. The full CPU run is intended as an overnight batch rather than an interactive check; the saved environment records the selected accelerator.

**Predefined outcome:** feasible structural condition match over all attempts. A positive paired seed-level 95% interval and an absolute effect of at least 0.05 are the full-run evidence criterion. The single-seed smoke run cannot establish significance. State changes measure empirical stabilization, not mathematical convergence. The primary metric is computed from decoded graph structure; the training-split feasibility estimator is optional secondary diagnostics, not a held-out validity oracle.


## 0 — Experiment metadata

In [1]:
EXPERIMENT_NAME = "recurrent_energy_nodefield_ablation_v1"
SEEDS = [0, 1, 2, 3, 4]
RUN_FULL = True


## 1 — Imports and reproducibility
Repository helpers seed Python, NumPy and Torch. Deterministic algorithms are enabled where supported; MPS is seeded but is not bitwise deterministic for every kernel. Environment, selected accelerator, library versions, and resolved configurations are saved with each run. The validation-calibrated anytime study samples its decoder-isomorphism check at the configured stride (16 by default) and always includes the full-budget step.

In [2]:
from conditional_node_field_graph_generator.extensions.demo.recurrent_experiments import (
    RecurrentExperiment, summarize_results, plot_results, load_results,
    analysis_section, decision_report,
)
experiment = RecurrentExperiment(smoke=not RUN_FULL)
print(experiment.run_dir)
K_TRAIN = experiment.configs["recurrent_energy_annealed"]["model"]["recurrent_training_steps"]


artifact/recurrent_nodefield/20260905T142937-55b50647


## 2 — One canonical dataset
Use the existing cycle/path/star generator and fitted vectorizers. The cached 80/10/10 split, supervision and training-only preprocessing are shared by every model.

In [3]:
experiment.prepare()
print({name: len(indices) for name, indices in experiment.splits.items()})


Enabling RDKit 2026.03.4 jupyter extensions
{'train': 800, 'validation': 100, 'test': 100}


## 3 — Primary model matrix

| ID | Model | Memory | Corruption | Energy |
|---|---|---|---|---|
| A | Baseline | No | Existing fixed sigma | Yes |
| B | RENF | Yes | Constant | Yes |
| C | RENF | Yes | Annealed | Yes |
| D | Same checkpoint as C | Intervention-dependent | Annealed | Yes |

D is an alias, not another training run. Parameter counts are measured after setup. Raw RENF has extra parameters; the notebook does not claim parameter matching.


In [4]:
print({name: config["model"] for name, config in experiment.configs.items()})


{'baseline': {'latent_embedding_dimension': 64, 'number_of_transformer_layers': 1, 'transformer_attention_head_count': 4, 'transformer_dropout': 0.1, 'batch_size': 16, 'learning_rate': 0.0001, 'maximum_epochs': 250, 'early_stopping_patience': 20, 'early_stopping_min_delta': 0.0, 'node_field_sigma': 0.2, 'sampling_step_size': 0.05, 'langevin_noise_scale': 0.0, 'recurrent_training_steps': 8, 'recurrent_hidden_dimension': 64, 'recurrent_detach_interval': 4, 'recurrent_update_scale': 1.0, 'recurrent_initial_state': 'zeros', 'recurrent_state_normalization': True, 'recurrent_sigma_min': 0.02, 'recurrent_sigma_max': 0.2, 'recurrent_supervise_all_steps': True, 'recurrent_loss_discount': 1.0, 'node_embedding_svd_dimension': 32, 'graph_embedding_svd_dimension': 32, 'node_vectorizer_parallel': False, 'graph_vectorizer_parallel': False, 'feasibility_parallel': False, 'decoder_n_jobs': 1, 'decoder_solver_threads': 1, 'locality_horizon': 2, 'use_feasibility_filtering': False, 'feasibility_oracle_can

## 4 — Sanity checks before training
Abort on nonfinite loss or gradients. Check shapes, padding, hidden influence and score gradients. Parameter-sharing and finite-difference checks are covered by the prerequisite test suite.

In [5]:
experiment.sanity_checks()


{'model': 'baseline', 'loss': 32.3193473815918, 'score_norm': 27.58984375, 'hidden_norm': 0.0, 'gradient_norm': 3.6675853729248047, 'parameter_count': 175829, 'peak_gpu_memory': None}
{'model': 'recurrent_energy_constant', 'loss': 32.407649993896484, 'score_norm': 21.19623374938965, 'hidden_norm': 0.930501401424408, 'gradient_norm': 6.404819965362549, 'parameter_count': 196949, 'peak_gpu_memory': None}
{'model': 'recurrent_energy_annealed', 'loss': 656.2283325195312, 'score_norm': 21.19623374938965, 'hidden_norm': 0.930501401424408, 'gradient_norm': 8.349261283874512, 'parameter_count': 196949, 'peak_gpu_memory': None}


,model,loss,score_norm,hidden_norm,gradient_norm,parameter_count,peak_gpu_memory
0,baseline,32.319347,27.589844,0.000000,3.667585,175829,None
1,recurrent_energy_constant,32.407650,21.196234,0.930501,6.404820,196949,None
2,recurrent_energy_annealed,656.228333,21.196234,0.930501,8.349261,196949,None


## 5 — Train and retain checkpoints
Smoke: equal ten-epoch budgets. Full: identical validation selection and patience; all epoch checkpoints are kept for matched-update curriculum comparisons. No test data selects checkpoints.

In [6]:
experiment.train()


`Trainer.fit` stopped: `max_epochs=250` reached.


Trained baseline seed=0 updates=12500


`Trainer.fit` stopped: `max_epochs=250` reached.


Trained recurrent_energy_constant seed=0 updates=12500


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have man

...

Trained baseline seed=4 updates=12500


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have man

Trained recurrent_energy_constant seed=4 updates=12500


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have man

Trained recurrent_energy_annealed seed=4 updates=12500


## 6 — Fixed-depth comparison and recorded evaluation
Runs the selected smoke/full matrix. Failed decodes remain rows. Unaligned generated graphs use label distributions rather than node-wise label accuracy.

In [ ]:
experiment.evaluate()
summary = summarize_results(experiment.run_dir)
results, diagnostics = load_results(experiment.run_dir)
analysis_section(results, diagnostics, "fixed_depth", k_train=K_TRAIN)


Evaluated baseline seed=0 depth=1
Evaluated baseline seed=0 depth=2
Evaluated baseline seed=0 depth=4
Evaluated baseline seed=0 depth=8
Evaluated baseline seed=0 depth=16
Evaluated baseline seed=0 depth=32
Evaluated baseline seed=0 depth=64
Evaluated baseline seed=0 depth=128
Evaluated baseline seed=0 depth=256
Evaluated recurrent_energy_constant seed=0 depth=1
Evaluated recurrent_energy_constant seed=0 depth=2


## 7 — Inference-depth scaling
The same checkpoints are evaluated beyond training depth. Full depths extend to 256; stop on nonfinite states.

In [ ]:
analysis_section(results, diagnostics, "depth", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 8 — Hidden-state reset
Reset occurs before the zero-based designated evaluation. Compare head count errors and saved per-step quality around the intervention.

In [ ]:
analysis_section(results, diagnostics, "reset", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 9 — Hidden-state shuffle
Full mode runs three independent within-graph permutations at each reset fraction. Smoke mode leaves this analysis empty.

In [ ]:
analysis_section(results, diagnostics, "shuffle", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 10 — Observable-state destruction
Full mode includes the complete persistent/fresh-x × persistent/reset-h grid. The smoke matrix contains normal, midpoint resets, and fresh-x every step.

In [ ]:
analysis_section(results, diagnostics, "channels", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 11 — Training curriculum
Compare constant and annealed models under normal inference, fresh-x noise, and increased depth. Full mode also compares retained checkpoints at matched updates.

In [ ]:
analysis_section(results, diagnostics, "curriculum", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 12 — Memory versus repeated computation
Full mode resets h before every evaluation on the same trained checkpoint.

In [ ]:
analysis_section(results, diagnostics, "no_memory", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 13 — Inference noise
Fresh x is N(0, s²I), with s recorded explicitly. Replacement noise and Langevin noise use distinct controls. All inference x values use the model’s scaled feature space.

In [ ]:
analysis_section(results, diagnostics, "noise", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 14 — State stability
Inspect hidden delta, score norm, potential, and prediction changes. These are empirical diagnostics; potential need not decrease when memory changes.

In [ ]:
analysis_section(results, diagnostics, "stability", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 15 — Anytime computation
Full mode selects stopping thresholds using validation trajectories, then measures test steps saved and mean/worst quality loss. Decoder-unchanged stopping uses three consecutive unchanged solutions. This secondary experiment is not executed in smoke mode.

In [ ]:
analysis_section(results, diagnostics, "anytime", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 16 — Statistical tables
Full comparisons aggregate independent training seeds and use paired seed-level confidence intervals. A one-seed run has undefined across-seed standard deviations and confidence intervals.

In [ ]:
analysis_section(results, diagnostics, "statistics", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 17 — Figures
All values come from saved results, diagnostics and training history. Re-run this cell to regenerate the figures without retraining.

In [ ]:
plot_results(experiment.run_dir)


## 18 — Decision criteria
The predefined effect criterion is applied only to full, independent-seed comparisons. Memory intervention evidence remains an intervention-based interpretation. Preservation under fresh-x corruption is an open experimental question, not an expected result.

In [ ]:
decision_report(experiment.run_dir)
